In [11]:
#import library
import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime

# Import MoleculeACE
from MoleculeACE.benchmark.data_prep import split_data
from MoleculeACE.benchmark.utils import Data
from MoleculeACE.benchmark.const import Descriptors
from MoleculeACE.models import RF
from MoleculeACE.benchmark.utils import calc_rmse, calc_cliff_rmse


# Setup directories
PROJECT_ROOT = Path(os.getcwd()).resolve()
DATA_DIR = PROJECT_ROOT / 'Data'
RESULTS_DIR = PROJECT_ROOT / 'Results'
PROCESS_DIR = PROJECT_ROOT / 'Process_Steps'
# Create all directories
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
PROCESS_DIR.mkdir(exist_ok=True)

In [12]:
# Import MoleculeACE
import MoleculeACE.benchmark.data_prep

# Load your CSV file
input_file = 'CHEMBL217_final.csv'
df_raw = pd.read_csv(input_file)
print(f" Loaded: {input_file}")
print(f"  Shape: {df_raw.shape}")
print(f"  Columns: {df_raw.columns.tolist()}")
print(f"  Sample:\n{df_raw.head(3).to_string()}\n")

# Save step 1
step1_file = PROCESS_DIR / ' CHEMBL217_final.csv'
df_raw.to_csv(step1_file, index=False)

# Check bioactivity column
bioactivity_col = 'exp_mean [nM]' 

# Step 1: Remove NaN values
df_clean = df_raw.dropna(subset=[bioactivity_col]).copy()
print(f"\n✓ Removed NaN: {len(df_raw) - len(df_clean)} rows")

# Step 2: Remove zero and negative values (bioactivity must be > 0 for log conversion)
df_clean = df_clean[df_clean[bioactivity_col] > 0].copy()
print(f"✓ Removed zero/negative bioactivity: {len(df_raw) - len(df_clean)} rows")

# Step 3: Check for SMILES column
if 'smiles' not in df_clean.columns:
    print("✗ ERROR: 'smiles' column not found!")
    print(f"Available columns: {df_clean.columns.tolist()}")
    raise ValueError("Missing 'smiles' column")

# Step 4: Remove invalid SMILES (empty strings, NaN)
df_clean = df_clean[df_clean['smiles'].notna()].copy()
df_clean = df_clean[df_clean['smiles'].str.strip() != ''].copy()
print(f"✓ Removed invalid SMILES: {len(df_raw) - len(df_clean)} rows total")
print(f"\nAfter cleaning:")
print(f"  Shape: {df_clean.shape}")
print(f"  Bioactivity stats:\n{df_clean[bioactivity_col].describe()}")
print(f"  Min bioactivity: {df_clean[bioactivity_col].min():.2e} nM")
print(f"  Max bioactivity: {df_clean[bioactivity_col].max():.2e} nM")
# Save cleaned data
clean_file = PROCESS_DIR / 'CHEMBL217_cleaned.csv'
df_clean.to_csv(clean_file, index=False)
print(f"\n✓ Cleaned data saved: {clean_file}\n")

# Extract SMILES and bioactivity
smiles_list = df_clean['smiles'].tolist()
bioactivity = df_clean['exp_mean [nM]'].tolist()

print(f" Extracted {len(smiles_list)} molecules")
print(f"  Bioactivity range: {min(bioactivity):.2e} - {max(bioactivity):.2e} nM\n")

df_prepared = MoleculeACE.benchmark.data_prep.split_data(
    smiles=smiles_list,
    bioactivity=bioactivity,  # ← Just the values, not the column name!
    n_clusters=5,
    test_size=0.2,
    similarity=0.9,
    potency_fold=10,
    remove_stereo=True
)

prepared_file = DATA_DIR / 'CHEMBL217_prepared.csv'
df_prepared.to_csv(prepared_file, index=False)

print(f" Data prepared!")
print(f"  Total: {len(df_prepared)}")
print(f"  Train: {(df_prepared['split'] == 'train').sum()}")
print(f"  Test: {(df_prepared['split'] == 'test').sum()}")
print(f"  Cliff molecules: {df_prepared['cliff_mol'].sum()}")



 Loaded: CHEMBL217_final.csv
  Shape: (7049, 4)
  Columns: ['smiles', 'chembl_id', 'exp_mean [nM]', 'class']
  Sample:
                                                 smiles     chembl_id  exp_mean [nM]  class
0     CC1Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3  CHEMBL349833          139.0  False
1  CC1(C)Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3  CHEMBL439646          201.0  False
2      Nc1cccc(-c2ccc(CCN3CCN(c4cccc5cccnc45)CC3)cc2)n1   CHEMBL87187           11.0   True


✓ Removed NaN: 0 rows
✓ Removed zero/negative bioactivity: 0 rows
✓ Removed invalid SMILES: 0 rows total

After cleaning:
  Shape: (7049, 4)
  Bioactivity stats:
count      7049.000000
mean       1277.730838
std        5423.432921
min           0.027000
25%          34.000000
50%         200.000000
75%         789.000000
max      180375.000000
Name: exp_mean [nM], dtype: float64
  Min bioactivity: 2.70e-02 nM
  Max bioactivity: 1.80e+05 nM

✓ Cleaned data saved: C:\Users\DELL\Downloads\Activity Cliff_proje

[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerator
[15:47:29] DEPRECATION WARNING: please use MorganGenerat

Removed 860 stereoisomers


[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerator
[15:48:54] DEPRECATION WARNING: please use MorganGenerat

Could not create a generic scaffold of Cc1ncoc1-c1nnc(SCCCN2CC3CC3(c3ccc(S(F)(F)(F)(F)F)cc3)C2)n1C, used a normal scaffold instead


[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerator
[15:49:58] DEPRECATION WARNING: please use MorganGenerat

Could not create a generic scaffold of Cc1cc(NC(=O)c2cccc(S(F)(F)(F)(F)F)c2)cc(-n2ccn3nc(-c4cccnc4)cc23)c1, used a normal scaffold instead


[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerator
[15:50:01] DEPRECATION WARNING: please use MorganGenerat

 Data prepared!
  Total: 6189
  Train: 4950
  Test: 1239
  Cliff molecules: 2252


In [13]:
# STEP 2: Load and Featurize

data = Data(str(prepared_file))

data.featurize_data(Descriptors.ECFP, radius=2, nbits=1024)

print(f" Featurized!")
print(f"  Descriptor: WHIM (Morgan Fingerprints)")
print(f"  Radius: 2")
print(f"  nBits: 1024")
print(f"  Train features: {data.x_train.shape}")
print(f"  Test features: {data.x_test.shape}\n")

#Train Model
model = RF(n_estimators=100, random_state=42)
model.train(data.x_train, data.y_train, None, None)
print(f" Model trained!")
print(f"  Algorithm: Random Forest")
print(f"  n_estimators: 100")
print(f"  Training samples: {len(data.y_train)}\n")

# STEP 3: PREDICTIONS ON BOTH TRAIN & TEST
# Predict on TRAIN data
y_pred_train = model.predict(data.x_train)
print(f"✓ Train predictions: {len(y_pred_train)} samples")

# Predict on TEST data
y_pred_test = model.predict(data.x_test)
print(f"✓ Test predictions: {len(y_pred_test)} samples\n")

# Train metrics
rmse_train = calc_rmse(data.y_train, y_pred_train)
rmse_cliff_train = calc_cliff_rmse(
    y_test_pred=y_pred_train,
    y_test=data.y_train,
    cliff_mols_test=data.cliff_mols_train
)
mae_train = np.mean(np.abs(np.array(data.y_train) - np.array(y_pred_train)))
r2_train = 1 - (np.sum((np.array(data.y_train) - np.array(y_pred_train))**2) / 
                 np.sum((np.array(data.y_train) - np.mean(data.y_train))**2))

# Test metrics
rmse_test = calc_rmse(data.y_test, y_pred_test)
rmse_cliff_test = calc_cliff_rmse(
    y_test_pred=y_pred_test,
    y_test=data.y_test,
    cliff_mols_test=data.cliff_mols_test
)
mae_test = np.mean(np.abs(np.array(data.y_test) - np.array(y_pred_test)))
r2_test = 1 - (np.sum((np.array(data.y_test) - np.array(y_pred_test))**2) / 
                np.sum((np.array(data.y_test) - np.mean(data.y_test))**2))

print(f" TRAIN METRICS:")
print(f"  RMSE: {rmse_train:.6f}")
print(f"  RMSE (Cliffs): {rmse_cliff_train:.6f}")
print(f"  MAE: {mae_train:.6f}")
print(f"  R²: {r2_train:.6f}")

print(f"\n TEST METRICS:")
print(f"  RMSE: {rmse_test:.6f}")
print(f"  RMSE (Cliffs): {rmse_cliff_test:.6f}")
print(f"  MAE: {mae_test:.6f}")
print(f"  R²: {r2_test:.6f}\n")


[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerator
[15:53:08] DEPRECATION WARNING: please use MorganGenerat

 Featurized!
  Descriptor: WHIM (Morgan Fingerprints)
  Radius: 2
  nBits: 1024
  Train features: (4950, 1024)
  Test features: (1239, 1024)

 Model trained!
  Algorithm: Random Forest
  n_estimators: 100
  Training samples: 4950

✓ Train predictions: 4950 samples
✓ Test predictions: 1239 samples

 TRAIN METRICS:
  RMSE: 0.229410
  RMSE (Cliffs): 0.267581
  MAE: 0.164762
  R²: 0.947102

 TEST METRICS:
  RMSE: 0.583143
  RMSE (Cliffs): 0.683087
  MAE: 0.429863
  R²: 0.664364



In [14]:
#Build Results Function
def build_results(df_split, smiles, cliff_mols, y_true, y_pred, split_name):
    """
    Build results dataframe with all columns including pEC50/pKi
    
    Args:
        df_split: DataFrame with split data (train or test)
        smiles: List of SMILES strings
        cliff_mols: List of cliff molecule indicators
        y_true: True y values
        y_pred: Predicted y values
        split_name: 'train' or 'test'
    
    Returns:
        DataFrame with results
    """
    
    # Safety check
    assert len(df_split) == len(smiles) == len(y_pred), \
        f"Length mismatch in {split_name} split: {len(df_split)} vs {len(smiles)} vs {len(y_pred)}"
    
    results = pd.DataFrame({
        "smiles": smiles,
        "exp_mean [nM]": df_split["exp_mean [nM]"].values,
        "y": df_split["y"].values,
        "cliff_mol": cliff_mols,
        "split": split_name,
    })
    
    # Calculate pEC50 / pKi
    print(f"  Calculating y [exp_mean [nM]] for {split_name}...")
    results["y[exp_mean [nM]"] = (
        9 - np.log10(results["exp_mean [nM]"].astype(float))
    )
    
    # Predictions & errors
    results["Predicted_y"] = y_pred
    results["Error"] = np.array(y_true) - np.array(y_pred)
    results["Abs_Error"] = np.abs(results["Error"])
    
    return results


# Get train data
df_train = (
    df_prepared[df_prepared["split"] == "train"]
    .reset_index(drop=True)
)

print(f" Building TRAIN results...")
train_results = build_results(
    df_split=df_train,
    smiles=data.smiles_train,
    cliff_mols=data.cliff_mols_train,
    y_true=data.y_train,
    y_pred=y_pred_train,
    split_name="train"
)
print(f"  Train results: {len(train_results)} rows")

# Get test data
df_test = (
    df_prepared[df_prepared["split"] == "test"]
    .reset_index(drop=True)
)

print(f"✓ Building TEST results...")
test_results = build_results(
    df_split=df_test,
    smiles=data.smiles_test,
    cliff_mols=data.cliff_mols_test,
    y_true=data.y_test,
    y_pred=y_pred_test,
    split_name="test"
)
print(f"  Test results: {len(test_results)} rows\n")

# Concatenate train and test
final_results = (
    pd.concat([train_results, test_results], ignore_index=True)
    .sort_values(["split", "Abs_Error"], ascending=[True, False])
    .reset_index(drop=True)
)

# Save complete results
final_results_file = RESULTS_DIR / 'CHEMBL217_predictions.csv'
final_results.to_csv(final_results_file, index=False)
print(f"✓ Saved: {final_results_file}")

 Building TRAIN results...
  Calculating y [exp_mean [nM]] for train...
  Train results: 4950 rows
✓ Building TEST results...
  Calculating y [exp_mean [nM]] for test...
  Test results: 1239 rows

✓ Saved: C:\Users\DELL\Downloads\Activity Cliff_project documents\XIAP_data\Results\CHEMBL217_predictions.csv


In [15]:
# Total molecules
n_total = len(final_results)
n_train = len(train_results)
n_test = len(test_results)

# Cliff molecules
cliff_total = final_results['cliff_mol'].sum()
cliff_train = train_results['cliff_mol'].sum()
cliff_test = test_results['cliff_mol'].sum()

# Percentages
percent_cliff_total = (cliff_total / n_total) * 100 if n_total > 0 else 0
percent_cliff_test = (cliff_test / n_test) * 100 if n_test > 0 else 0

# Create statistics dataframe
statistics = pd.DataFrame({
    'Metric': [
        'n (Total molecules)',
        'nTRAIN (Train molecules)',
        'nTEST (Test molecules)',
        'Cliff (Total cliff molecules)',
        'Cliff TRAIN (Train cliff molecules)',
        'Cliff TEST (Test cliff molecules)',
        '%cliff (% of cliffs in full dataset)',
        '%cliffTRAIN (% of cliffs in train set)',
        '%cliffTEST (% of cliffs in test set)'
    ],
    'Value': [
        n_total,
        n_train,
        n_test,
        cliff_total,
        cliff_train,
        cliff_test,
        f"{percent_cliff_total:.2f}",
        f"{(cliff_train / n_train * 100):.2f}" if n_train > 0 else 0,
        f"{percent_cliff_test:.2f}"
    ]
})

print(f"✓ Statistics calculated:")
print(statistics.to_string(index=False))
print()

# Save as CSV
stats_csv = RESULTS_DIR / 'CHEMBL217_statistics.csv'
statistics.to_csv(stats_csv, index=False)
print(f"✓ Saved (CSV): {stats_csv}\n")

✓ Statistics calculated:
                                Metric Value
                   n (Total molecules)  6189
              nTRAIN (Train molecules)  4950
                nTEST (Test molecules)  1239
         Cliff (Total cliff molecules)  2252
   Cliff TRAIN (Train cliff molecules)  1802
     Cliff TEST (Test cliff molecules)   450
  %cliff (% of cliffs in full dataset) 36.39
%cliffTRAIN (% of cliffs in train set) 36.40
  %cliffTEST (% of cliffs in test set) 36.32

✓ Saved (CSV): C:\Users\DELL\Downloads\Activity Cliff_project documents\XIAP_data\Results\CHEMBL217_statistics.csv

